# Tệp: notebooks/moocradar_colab.ipynb
# Mục đích: Tái lập bước chuẩn hóa, huấn luyện, đánh giá và lộ trình học trên Google Colab.

Đặt repository và bốn file raw trên Google Drive. Kiểm tra các đường dẫn bên dưới rồi chạy cell theo thứ tự. Có thể chọn `bkt` hoặc `ncdm`; baseline luôn được giữ trong run. `smoke_test` dùng 25 sinh viên để kiểm tra nhanh, còn `full_run` dùng toàn bộ interaction. Artifacts và checkpoint được lưu trực tiếp trên Drive. Nên chạy notebook [Data Quality & EDA](01_data_quality_eda.ipynb) trước bước huấn luyện.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import os
import subprocess

PROJECT_ROOT = Path("/content/drive/MyDrive/Personalized_learning_path")
DATA_ROOT = PROJECT_ROOT / "data/raw"
PROCESSED_ROOT = PROJECT_ROOT / "data/processed"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
MODE = "smoke_test"  # Change to "full_run" after reviewing the smoke outputs.
MODEL = "ncdm"  # "bkt" or "ncdm"; the smoothed baseline is always retained.
GRANULARITY = "middle"  # Recommended default for NCDM on Colab.
HIDDEN_DIMS = "256,128"  # Use "64,32" for the documented CPU full run.
DROPOUT = "0.5"
TRAIN = True  # Set False and provide SAVED_RUN_DIR to reuse a checkpoint.
SAVED_RUN_DIR = None  # Example: ARTIFACT_ROOT / "20261020T..._ncdm_..."
assert (PROJECT_ROOT / "requirements.txt").exists(), "Set PROJECT_ROOT to the repository folder."
assert (DATA_ROOT / "problem.json").exists(), "Place the raw files in DATA_ROOT."
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")], check=True)


In [ ]:
import shutil
print("Số lõi CPU:", os.cpu_count())
print("RAM (GiB):", round(os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE") / 2**30, 2))
print("Dung lượng trống (GiB):", round(shutil.disk_usage("/content").free / 2**30, 2))
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không có; pipeline hỗ trợ CPU")
except ImportError:
    print("GPU check không có; pipeline hỗ trợ CPU")


## Chuẩn bị dữ liệu

Bước này đọc JSON array theo từng chuỗi, kiểm tra chất lượng và lưu các bảng chuẩn hóa. Run mẫu vẫn đọc đầy đủ problem metadata. Dữ liệu gốc không bị chỉnh sửa.


In [ ]:
processed_dir = PROCESSED_ROOT / ("smoke" if MODE == "smoke_test" else "full")
env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT / "src"), "MPLCONFIGDIR": "/content/.mplconfig"}
prepare_cmd = ["python", "-m", "moocradar", "prepare", "--data-root", str(DATA_ROOT), "--output-dir", str(processed_dir)]
if MODE == "smoke_test":
    prepare_cmd += ["--max-students", "25"]
subprocess.run(prepare_cmd, cwd=PROJECT_ROOT, env=env, check=True)


## Huấn luyện, đánh giá và xuất hồ sơ/lộ trình

Huấn luyện tạo thư mục run mới. NCDM tự chọn CUDA nếu có và fallback về CPU, in validation AUC/Brier mỗi epoch, early stopping và lưu `best_ncdm.pt`. Đặt `TRAIN=False` để load một run đã lưu trên Drive mà không train lại. BKT vẫn có checkpoint JSON và khả năng cập nhật tuần tự riêng.


In [ ]:
if TRAIN:
    before = set(ARTIFACT_ROOT.glob("*/manifest.json"))
    train_cmd = ["python", "-m", "moocradar", "train", "--model", MODEL, "--processed-dir", str(processed_dir), "--artifact-root", str(ARTIFACT_ROOT), "--seed", "42"]
    if MODEL == "ncdm":
        train_cmd += ["--granularity", GRANULARITY, "--batch-size", "512", "--learning-rate", "0.002", "--epochs", "3" if MODE == "smoke_test" else "20", "--patience", "4", "--hidden-dims", HIDDEN_DIMS, "--dropout", DROPOUT, "--device", "auto"]
    subprocess.run(train_cmd, cwd=PROJECT_ROOT, env=env, check=True)
    new_manifests = set(ARTIFACT_ROOT.glob("*/manifest.json")) - before
    assert len(new_manifests) == 1
    run_dir = next(iter(new_manifests)).parent
else:
    assert SAVED_RUN_DIR is not None, "Set SAVED_RUN_DIR when TRAIN=False"
    run_dir = Path(SAVED_RUN_DIR)
print("Run đã lưu:", run_dir)


In [ ]:
import csv
import json
manifest = json.loads((run_dir / "manifest.json").read_text())
assert manifest["status"] == "complete"
with (run_dir / "metrics.csv").open() as f:
    for row in csv.DictReader(f):
        print(row)
required = ["data_quality.json", "predictions.csv", "metrics.csv", "student_mastery.csv", "cognitive_profile.csv", "knowledge_gaps.csv", "learning_paths.csv", "models/bkt.json"]
if MODEL == "ncdm":
    required += ["models/best_ncdm.pt", "models/id_mappings.json", "models/q_matrix.npz", "plots/training_loss.png", "plots/roc_curve.png"]
for name in required:
    assert (run_dir / name).exists(), name
print("Hồ sơ và lộ trình:", run_dir / "student_mastery.csv", run_dir / "learning_paths.csv")


## Demo JSON và chia sẻ kết quả

Chọn một sinh viên từ bảng interaction và load checkpoint để xuất JSON. BKT có thể nhận thêm một response để cập nhật trạng thái tuần tự. NCDM chuẩn chỉ suy luận từ checkpoint đóng băng; muốn hấp thụ dữ liệu mới phải retrain định kỳ. Tải cả thư mục run trên Drive để bàn giao cho dashboard.


In [ ]:
with (processed_dir / "interactions.csv").open() as f:
    user_id = next(csv.DictReader(f))["user_id"]
subprocess.run(["python", "-m", "moocradar", "demo", "--model", MODEL, "--run-dir", str(run_dir), "--processed-dir", str(processed_dir), "--user-id", user_id], cwd=PROJECT_ROOT, env=env, check=True)
print("Chia sẻ thư mục Drive này:", run_dir)
